# 02. Score Quality

`papers_raw.csv`를 받아 인용수, 연식 보정 인용수, 최신성, venue, abstract, 교육공학 관련성을 함께 점수화합니다.


In [1]:
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_raw.csv')
print(f'{len(df)} papers loaded')


50 papers loaded


In [2]:
FROM_YEAR = 2020
TO_YEAR = 2026
KEY_TERMS = [
    'educational technology', 'artificial intelligence', 'generative ai', 'chatgpt',
    'ai feedback', 'formative feedback', 'self-regulated learning', 'self regulated learning',
    'learning analytics', 'adaptive learning', 'intelligent tutoring',
]

def keyword_hits(row):
    text = f"{row.get('title', '')} {row.get('abstract', '')}".lower()
    hits = [term for term in KEY_TERMS if term in text]
    if re.search(r'ai', text) and 'ai feedback' not in hits and 'generative ai' not in hits:
        hits.append('AI')
    return hits

def percentile(series):
    series = series.fillna(0).astype(float)
    if series.max() == series.min():
        return pd.Series([0.5] * len(series), index=series.index)
    return series.rank(pct=True)

scored = df.copy()
scored['keyword_hits'] = scored.apply(keyword_hits, axis=1)
scored['keyword_hit_count'] = scored['keyword_hits'].apply(len)
scored['topic_relevance_score'] = (scored['keyword_hit_count'] / 4).clip(upper=1).round(3)

year = scored['year'].fillna(FROM_YEAR).astype(float)
recency = ((year - FROM_YEAR) / max(TO_YEAR - FROM_YEAR, 1)).clip(0, 1)

scored['quality_score'] = (
    percentile(scored['cited_by_count']) * 0.25
    + percentile(scored['citations_per_year']) * 0.25
    + scored['topic_relevance_score'] * 0.25
    + recency * 0.15
    + scored['venue'].notna().astype(float) * 0.05
    + scored['abstract'].fillna('').str.len().ge(200).astype(float) * 0.05
).round(3)

def role(row):
    if row['topic_relevance_score'] >= 0.75 and row['quality_score'] >= 0.70:
        return 'core_candidate'
    if row['topic_relevance_score'] >= 0.50 and row['quality_score'] >= 0.55:
        return 'supporting_candidate'
    if row['topic_relevance_score'] >= 0.25:
        return 'background_candidate'
    return 'screen_out_candidate'

scored['candidate_role'] = scored.apply(role, axis=1)
scored['keyword_hits'] = scored['keyword_hits'].apply(lambda xs: '; '.join(xs))
df_sorted = scored.sort_values('quality_score', ascending=False).reset_index(drop=True)
df_sorted[['title', 'year', 'cited_by_count', 'topic_relevance_score', 'quality_score', 'candidate_role']].head(15)


,title,year,cited_by_count,topic_relevance_score,quality_score,candidate_role
0,How Generative AI Influences Students’ Self-Re...,2025,71,1.00,0.950,core_candidate
1,Educational Design Principles of Using AI Chat...,2023,274,1.00,0.915,core_candidate
2,Beware of metacognitive laziness: Effects of g...,2024,348,0.75,0.888,core_candidate
3,Hybrid intelligence: Human– AI coevolution and...,2025,35,0.75,0.868,core_candidate
4,Self-Regulated Learning in the Digital Age: A ...,2025,32,0.75,0.858,core_candidate
5,A conceptual exploration of generative AI-indu...,2025,11,1.00,0.850,core_candidate
6,Has artificial intelligence rendered language ...,2024,16,1.00,0.838,core_candidate
7,Adapting educational practices for Generation ...,2025,39,0.50,0.815,supporting_candidate
8,Using Artificial Intelligence for Higher Educa...,2025,14,0.75,0.808,core_candidate
9,Enhancing legal writing skills: The impact of ...,2024,27,0.75,0.798,core_candidate


In [3]:
out = DATA_DIR / 'papers_scored.csv'
df_sorted.to_csv(out, index=False)
print(f'Saved -> {out.resolve()}')
print(df_sorted['candidate_role'].value_counts())


Saved -> /Users/sungjae-cha/Documents/research-agent/data/papers_scored.csv
candidate_role
background_candidate    19
supporting_candidate    18
core_candidate          13
Name: count, dtype: int64
